# 05b - Evaluate Strategy Suites

CPU only. Evaluate all eleven strategy themes with explicit label coverage. Validation suites support selection; the frozen winner's test suites, including held-out annotated games, run in 06a.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Load the Same Candidates

Load the common registry independently of 05a; no match results are required. It includes the independent classical control and any configured no-RL runs. For neural models, top-k compares legal policy rankings; move accuracy compares the search-selected move. Classical top-k is unavailable and value is a heuristic.

In [ ]:
import torch
torch.set_num_threads(1)
from chess_rl.candidate_registry import load_candidate_registry
registry = load_candidate_registry(PROJECT_ROOT, cfg)
candidates = registry["candidates"]
print("Evaluation split:", cfg["strategy_evaluation"]["split"])

## Evaluate Themed Suites

Report move accuracy, top-k agreement, value MAE, legal move rate, prediction failures and per-position/per-theme pass/fail/not_assessable counts. Neural policy top-1 and top-3 are separate from search agreement. History-dependent cases are reported separately. Acceptable move sets are validated; missing labels are not failures. Heuristics are not gold labels.

In [ ]:
from chess_rl.strategy_evaluation import evaluate_candidates
strategy = evaluate_candidates(PROJECT_ROOT, cfg, candidates)
from chess_rl.strategy_plots import evaluation_summary
display(evaluation_summary(PROJECT_ROOT, cfg["run_id"], strategy))

## Persistent Results

Per-position observations, summary JSON/CSV and a plot are saved together.

In [ ]:
print(PROJECT_ROOT / "results/strategy_evaluation" / cfg["run_id"] / cfg["strategy_evaluation"]["split"])
print("Next: 06a_select_and_freeze_winner.ipynb.")